Trying to recreate the preprocessing process of the core paper experiment.

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

# Change these paths if your CSV files are somewhere else
DATA_PATH = Path("Raw_Dataset/diabetic_data.csv")
MAP_PATH = Path("Raw_Dataset/IDS_mapping.csv")

# Important:
# keep_default_na=False keeps the string "None" in A1Cresult.
# In this dataset, "None" means "test was not measured", not a missing Python value.
df = pd.read_csv(
    DATA_PATH,
    na_values=["?"],
    keep_default_na=False,
    low_memory=False
)

ids_mapping = pd.read_csv(MAP_PATH, keep_default_na=False)

print(df.shape)
df.head()

(101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [5]:
clean = df.copy()

# 1. Drop columns the paper did not use because of too many missing values
clean = clean.drop(columns=["weight", "payer_code"])

# 2. Keep medical_specialty, but mark missing values clearly
clean["medical_specialty"] = clean["medical_specialty"].fillna("Missing")
clean["race"] = clean["race"].fillna("Missing")

# 3. Remove invalid gender rows
# There are only a few, so this is safe.
clean = clean[clean["gender"] != "Unknown/Invalid"].copy()

# 4. Remove death / hospice discharge cases
# From IDS_mapping:
# 11 = Expired
# 13 = Hospice / home
# 14 = Hospice / medical facility
# 19, 20, 21 = expired / hospice related
death_hospice_ids = [11, 13, 14, 19, 20, 21]

clean = clean[
    ~clean["discharge_disposition_id"].isin(death_hospice_ids)
].copy()

# 5. Keep only the first encounter for each patient
clean = (
    clean
    .sort_values(["patient_nbr", "encounter_id"])
    .drop_duplicates(subset="patient_nbr", keep="first")
)

# 6. Define binary 30-day readmission outcome
# 1 = readmitted within 30 days
# 0 = not readmitted within 30 days
clean["readmitted_30"] = (clean["readmitted"] == "<30").astype(int)

print(clean.shape)
print(clean["readmitted_30"].value_counts())
print(clean["readmitted_30"].mean())

(69987, 49)
readmitted_30
0    63702
1     6285
Name: count, dtype: int64
0.08980239187277637


In [6]:
def make_a1c_group(row):
    if row["A1Cresult"] == "None":
        return "Not measured"
    elif row["A1Cresult"] == ">8" and row["change"] == "Ch":
        return "High, medication changed"
    elif row["A1Cresult"] == ">8" and row["change"] == "No":
        return "High, medication not changed"
    else:
        # This includes "Norm" and ">7"
        # The paper's table has a "normal result" group, but the public dataset also has ">7".
        # We can document this later.
        return "Normal/near-normal measured"


def diagnosis_group(code):
    code = str(code).strip()

    if code == "" or code.lower() == "nan":
        return "Other"

    # ICD codes beginning with V or E are special codes.
    # The paper groups these into Other.
    if code.startswith("V") or code.startswith("E"):
        return "Other"

    try:
        num = float(code)
    except ValueError:
        return "Other"

    if (390 <= num <= 459) or num == 785:
        return "Circulatory"
    elif (460 <= num <= 519) or num == 786:
        return "Respiratory"
    elif (520 <= num <= 579) or num == 787:
        return "Digestive"
    elif code.startswith("250"):
        return "Diabetes"
    elif 800 <= num <= 999:
        return "Injury"
    elif 710 <= num <= 739:
        return "Musculoskeletal"
    elif (580 <= num <= 629) or num == 788:
        return "Genitourinary"
    elif 140 <= num <= 239:
        return "Neoplasms"
    else:
        return "Other"


def age_group(age):
    # age looks like "[50-60)"
    left = int(age.strip("[]()").split("-")[0])

    if left < 30:
        return "<30"
    elif left < 60:
        return "30-60"
    else:
        return "60+"


def specialty_group(s):
    if s == "Missing":
        return "Missing"
    elif s == "InternalMedicine":
        return "Internal Medicine"
    elif s == "Cardiology":
        return "Cardiology"
    elif s == "Family/GeneralPractice":
        return "Family/General Practice"
    elif s.startswith("Surgery"):
        return "Surgery"
    else:
        return "Other"


def race_group(r):
    if r == "AfricanAmerican":
        return "African American"
    elif r == "Caucasian":
        return "Caucasian"
    elif r == "Missing":
        return "Missing"
    else:
        return "Other"


clean["a1c_group"] = clean.apply(make_a1c_group, axis=1)
clean["diag1_group"] = clean["diag_1"].apply(diagnosis_group)
clean["age_group"] = clean["age"].apply(age_group)
clean["specialty_group"] = clean["medical_specialty"].apply(specialty_group)
clean["race_group"] = clean["race"].apply(race_group)

clean["admission_source_group"] = np.select(
    [
        clean["admission_source_id"].eq(7),
        clean["admission_source_id"].isin([1, 2, 3])
    ],
    [
        "Emergency",
        "Referral"
    ],
    default="Other"
)

clean["discharge_group"] = np.where(
    clean["discharge_disposition_id"].eq(1),
    "Home",
    "Other"
)

In [7]:
columns_to_check = [
    "a1c_group",
    "diag1_group",
    "age_group",
    "admission_source_group",
    "discharge_group",
    "specialty_group",
    "race_group"
]

for col in columns_to_check:
    print("\n", col)
    print(clean[col].value_counts())


 a1c_group
a1c_group
Not measured                    57141
Normal/near-normal measured      6607
High, medication changed         4058
High, medication not changed     2181
Name: count, dtype: int64

 diag1_group
diag1_group
Circulatory        21389
Other              12134
Respiratory         9491
Digestive           6488
Diabetes            5748
Injury              4694
Musculoskeletal     4064
Genitourinary       3441
Neoplasms           2538
Name: count, dtype: int64

 age_group
age_group
60+      46308
30-60    21871
<30       1808
Name: count, dtype: int64

 admission_source_group
admission_source_group
Emergency    37271
Referral     22792
Other         9924
Name: count, dtype: int64

 discharge_group
discharge_group
Home     44320
Other    25667
Name: count, dtype: int64

 specialty_group
specialty_group
Missing                    33652
Other                      12824
Internal Medicine          10641
Family/General Practice     4978
Cardiology                  4207
Surgery   

In [5]:
Path("processed").mkdir(exist_ok=True)

clean.to_csv("processed/diabetes_preprocessed_stage1.csv", index=False)

print("Saved cleaned dataset.")
print(clean.shape)

Saved cleaned dataset.
(69987, 56)


Preprocess done, move on to getting summary table

In [1]:
def summarise_categorical(df, column, variable_name=None, category_order=None):
    """
    Create a Table 3-style summary for one categorical variable.

    Output columns:
    - Variable
    - Group
    - Number of encounters
    - % of population
    - Readmitted encounters
    - % readmitted in group
    """
    if variable_name is None:
        variable_name = column

    temp = (
        df.groupby(column, dropna=False)["readmitted_30"]
        .agg(
            encounters="count",
            readmitted="sum",
            readmission_rate="mean"
        )
        .reset_index()
        .rename(columns={column: "Group"})
    )

    temp["Variable"] = variable_name
    temp["population_percent"] = temp["encounters"] / len(df) * 100
    temp["readmission_percent"] = temp["readmission_rate"] * 100

    temp = temp[
        [
            "Variable",
            "Group",
            "encounters",
            "population_percent",
            "readmitted",
            "readmission_percent"
        ]
    ]

    if category_order is not None:
        temp["Group"] = pd.Categorical(
            temp["Group"],
            categories=category_order,
            ordered=True
        )
        temp = temp.sort_values("Group")

    return temp

In [2]:
summary_specs = [
    {
        "column": "a1c_group",
        "name": "HbA1c",
        "order": [
            "Not measured",
            "High, medication changed",
            "High, medication not changed",
            "Normal/near-normal measured"
        ]
    },
    {
        "column": "gender",
        "name": "Gender",
        "order": [
            "Female",
            "Male"
        ]
    },
    {
        "column": "discharge_group",
        "name": "Discharge disposition",
        "order": [
            "Home",
            "Other"
        ]
    },
    {
        "column": "admission_source_group",
        "name": "Admission source",
        "order": [
            "Emergency",
            "Referral",
            "Other"
        ]
    },
    {
        "column": "specialty_group",
        "name": "Specialty of admitting physician",
        "order": [
            "Internal Medicine",
            "Cardiology",
            "Surgery",
            "Family/General Practice",
            "Missing",
            "Other"
        ]
    },
    {
        "column": "diag1_group",
        "name": "Primary diagnosis",
        "order": [
            "Circulatory",
            "Diabetes",
            "Respiratory",
            "Digestive",
            "Injury",
            "Musculoskeletal",
            "Genitourinary",
            "Neoplasms",
            "Other"
        ]
    },
    {
        "column": "race_group",
        "name": "Race",
        "order": [
            "African American",
            "Caucasian",
            "Other",
            "Missing"
        ]
    },
    {
        "column": "age_group",
        "name": "Age group",
        "order": [
            "<30",
            "30-60",
            "60+"
        ]
    }
]

In [8]:
summary_tables = []

for spec in summary_specs:
    table_part = summarise_categorical(
        clean,
        column=spec["column"],
        variable_name=spec["name"],
        category_order=spec["order"]
    )
    summary_tables.append(table_part)

table3_like = pd.concat(summary_tables, ignore_index=True)

table3_like.head(20)

,Variable,Group,encounters,population_percent,readmitted,readmission_percent
0,HbA1c,Not measured,57141,81.645163,5206,9.110796
1,HbA1c,"High, medication changed",4058,5.798220,348,8.575653
2,HbA1c,"High, medication not changed",2181,3.116293,161,7.381935
3,HbA1c,Normal/near-normal measured,6607,9.440325,570,8.627214
4,Gender,Female,37239,53.208453,3365,9.036225
5,Gender,Male,32748,46.791547,2920,8.916575
6,Discharge disposition,Home,44320,63.326046,3079,6.947202
7,Discharge disposition,Other,25667,36.673954,3206,12.490747
8,Admission source,Emergency,37271,53.254176,3452,9.261893
9,Admission source,Referral,22792,32.566048,1973,8.656546


In [9]:
table3_display = table3_like.copy()

table3_display["population_percent"] = table3_display["population_percent"].round(1)
table3_display["readmission_percent"] = table3_display["readmission_percent"].round(1)

table3_display = table3_display.rename(
    columns={
        "Variable": "Variable",
        "Group": "Group",
        "encounters": "Number of encounters",
        "population_percent": "% of population",
        "readmitted": "Readmitted encounters",
        "readmission_percent": "% readmitted in group"
    }
)

pd.set_option("display.max_rows", 100)
table3_display

,Variable,Group,Number of encounters,% of population,Readmitted encounters,% readmitted in group
0,HbA1c,Not measured,57141,81.6,5206,9.1
1,HbA1c,"High, medication changed",4058,5.8,348,8.6
2,HbA1c,"High, medication not changed",2181,3.1,161,7.4
3,HbA1c,Normal/near-normal measured,6607,9.4,570,8.6
4,Gender,Female,37239,53.2,3365,9.0
5,Gender,Male,32748,46.8,2920,8.9
6,Discharge disposition,Home,44320,63.3,3079,6.9
7,Discharge disposition,Other,25667,36.7,3206,12.5
8,Admission source,Emergency,37271,53.3,3452,9.3
9,Admission source,Referral,22792,32.6,1973,8.7


In [10]:
Path("outputs").mkdir(exist_ok=True)

table3_like.to_csv("outputs/table3_like_raw.csv", index=False)
table3_display.to_csv("outputs/table3_like_display.csv", index=False)

print("Saved:")
print("outputs/table3_like_raw.csv")
print("outputs/table3_like_display.csv")

Saved:
outputs/table3_like_raw.csv
outputs/table3_like_display.csv


In [11]:
def age_midpoint(age_range):
    """
    Convert age group like '[50-60)' into midpoint, for example 55.
    """
    left, right = age_range.strip("[]()").split("-")
    return (int(left) + int(right)) / 2


clean["age_midpoint"] = clean["age"].apply(age_midpoint)

continuous_summary = pd.DataFrame(
    {
        "Variable": ["Age midpoint", "Time in hospital"],
        "Mean": [
            clean["age_midpoint"].mean(),
            clean["time_in_hospital"].mean()
        ],
        "Median": [
            clean["age_midpoint"].median(),
            clean["time_in_hospital"].median()
        ],
        "1st quartile": [
            clean["age_midpoint"].quantile(0.25),
            clean["time_in_hospital"].quantile(0.25)
        ],
        "3rd quartile": [
            clean["age_midpoint"].quantile(0.75),
            clean["time_in_hospital"].quantile(0.75)
        ]
    }
)

continuous_summary = continuous_summary.round(2)
continuous_summary

,Variable,Mean,Median,1st quartile,3rd quartile
0,Age midpoint,65.44,65.0,55.0,75.0
1,Time in hospital,4.27,3.0,2.0,6.0


In [12]:
continuous_summary.to_csv("outputs/table3_continuous_summary.csv", index=False)

In [13]:
table3_display[table3_display["Variable"] == "HbA1c"]

,Variable,Group,Number of encounters,% of population,Readmitted encounters,% readmitted in group
0,HbA1c,Not measured,57141,81.6,5206,9.1
1,HbA1c,"High, medication changed",4058,5.8,348,8.6
2,HbA1c,"High, medication not changed",2181,3.1,161,7.4
3,HbA1c,Normal/near-normal measured,6607,9.4,570,8.6


In [14]:
table3_display[table3_display["Variable"] == "Primary diagnosis"]

,Variable,Group,Number of encounters,% of population,Readmitted encounters,% readmitted in group
17,Primary diagnosis,Circulatory,21389,30.6,2070,9.7
18,Primary diagnosis,Diabetes,5748,8.2,524,9.1
19,Primary diagnosis,Respiratory,9491,13.6,693,7.3
20,Primary diagnosis,Digestive,6488,9.3,520,8.0
21,Primary diagnosis,Injury,4694,6.7,507,10.8
22,Primary diagnosis,Musculoskeletal,4064,5.8,341,8.4
23,Primary diagnosis,Genitourinary,3441,4.9,309,9.0
24,Primary diagnosis,Neoplasms,2538,3.6,230,9.1
25,Primary diagnosis,Other,12134,17.3,1091,9.0
